In [1]:
# ============================================================
# Dynamic Marginal-Utility Meal Bundles + M1 Ultra Metric
# - Loads your 2 CSVs
# - Prints dataset structure + samples
# - Builds meal->categories map and user->required_categories
# - Runs dynamic marginal-utility selection (bundle of K meals)
# - Evaluates each bundle with M1.apply_ultra_metric
# - Saves results to CSV
# ============================================================

import pandas as pd
import numpy as np
import ast, math, random
from collections import Counter
import os, datetime

# ----------------------------
# 0) CONFIG
# ----------------------------
REQ_PATH = "../data/input_data/user_meal_requirements.csv"
MEAL_PATH = "../data/input_data/meal_categories.csv"

OUT_DIR = "../data/meal_knapsack_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Bundle size (how many meals to recommend per user row)
K_MEALS = 3

# Diminishing returns base:
# 1st coverage: 1.0, 2nd: base, 3rd: base^2 ...
DIM_BASE = 0.5

# Optional "seat cost" to discourage adding meals with weak marginal utility
COST_CONSTANT = 0.0



In [2]:
# ----------------------------
# 1) IMPORT M1
# ----------------------------
import M1
# If ModuleNotFoundError, add your folder containing M1.py:
# import sys
# sys.path.append("/path/to/folder/with/M1.py")
# import M1

print("Loaded M1. Using M1.apply_ultra_metric(req_skills, team, pseudo_skills_map)")



Loaded M1. Using M1.apply_ultra_metric(req_skills, team, pseudo_skills_map)


[nltk_data] Downloading package wordnet to /Users/tej/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/tej/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [3]:
# ----------------------------
# 2) HELPERS
# ----------------------------
def parse_set(x):
    """Parse a string like "['a','b']" or "{a,b}" into a Python set."""
    if pd.isna(x):
        return set()
    if isinstance(x, (set, list, tuple)):
        return set(x)
    try:
        v = ast.literal_eval(x)
        if isinstance(v, (set, list, tuple)):
            return set(v)
        return set()
    except Exception:
        s = str(x).strip()
        s = s.strip("{}")
        parts = [p.strip().strip("'").strip('"') for p in s.split(",") if p.strip()]
        return set(parts)

def coverage_pct(req, covered):
    return 100.0 * len(covered) / max(1, len(req))



In [4]:
# ----------------------------
# 3) LOAD DATA + PRINT STRUCTURE
# ----------------------------
print("\n=== Paths ===")
print("REQ_PATH :", REQ_PATH)
print("MEAL_PATH:", MEAL_PATH)

req_df  = pd.read_csv(REQ_PATH)
meal_df = pd.read_csv(MEAL_PATH)

print("\n=== Shapes ===")
print("user_meal_requirements:", req_df.shape)
print("meal_categories      :", meal_df.shape)

print("\n=== Columns (user requirements) ===")
print(req_df.columns.tolist())

print("\n=== Columns (meal categories) ===")
print(meal_df.columns.tolist())

print("\n=== Head (user requirements) ===")
print(req_df.head(5).to_string(index=False))

print("\n=== Head (meal categories) ===")
print(meal_df.head(5).to_string(index=False))

print("\n=== Info (user requirements) ===")
print(req_df.info())

print("\n=== Info (meal categories) ===")
print(meal_df.info())





=== Paths ===
REQ_PATH : ../data/input_data/user_meal_requirements.csv
MEAL_PATH: ../data/input_data/meal_categories.csv

=== Shapes ===
user_meal_requirements: (500, 4)
meal_categories      : (2000, 3)

=== Columns (user requirements) ===
['Unnamed: 0', 'user_id', 'meal_occasion', 'required_categories']

=== Columns (meal categories) ===
['Unnamed: 0', 'meal_name', 'categories']

=== Head (user requirements) ===
 Unnamed: 0          user_id meal_occasion                    required_categories
          0 user_0_breakfast     breakfast         {'Breakfast Item', 'Beverage'}
          1 user_1_breakfast     breakfast {'Breakfast Item', 'Beverage', 'Side'}
          2 user_2_breakfast     breakfast         {'Breakfast Item', 'Beverage'}
          3 user_3_breakfast     breakfast            {'Main Course', 'Beverage'}
          4     user_4_lunch         lunch          {'Salad', 'Soup', 'Beverage'}

=== Head (meal categories) ===
 Unnamed: 0                    meal_name                  

In [5]:
# ----------------------------
# 4) EXPECTED COLUMN NAMES (EDIT HERE IF NEEDED)
# ----------------------------
# Adjust these 3 if your CSV uses different column names.
USER_ID_COL        = "user_id"
OCCASION_COL       = "meal_occasion"
REQUIRED_CATS_COL  = "required_categories"

MEAL_NAME_COL      = "meal_name"
MEAL_CATS_COL      = "categories"

# quick sanity check
for col in [USER_ID_COL, OCCASION_COL, REQUIRED_CATS_COL]:
    if col not in req_df.columns:
        raise ValueError(f"Missing column in user_meal_requirements.csv: {col}")

for col in [MEAL_NAME_COL, MEAL_CATS_COL]:
    if col not in meal_df.columns:
        raise ValueError(f"Missing column in meal_categories.csv: {col}")



In [6]:
# ----------------------------
# 5) PARSE SETS + BUILD MAPS
# ----------------------------
req_df["req_set"] = req_df[REQUIRED_CATS_COL].apply(parse_set)
meal_df["cat_set"] = meal_df[MEAL_CATS_COL].apply(parse_set)

meal_to_cats = dict(zip(meal_df[MEAL_NAME_COL], meal_df["cat_set"]))

all_meals = list(meal_to_cats.keys())

print("\n=== Parsed Sets ===")
print("Num meals:", len(all_meals))
print("Example meal:", all_meals[0])
print("Example meal categories:", list(meal_to_cats[all_meals[0]])[:15])

print("\nExample requirement row:")
ex = req_df.iloc[0]
print("User:", ex[USER_ID_COL], "| Occasion:", ex[OCCASION_COL])
print("Required categories:", ex["req_set"])




=== Parsed Sets ===
Num meals: 2000
Example meal: Yogurt Classic (m1)
Example meal categories: ['Breakfast Item']

Example requirement row:
User: user_0_breakfast | Occasion: breakfast
Required categories: {'Breakfast Item', 'Beverage'}


In [7]:
# ----------------------------
# 6) CATEGORY WEIGHTS (IDF across meals)
# ----------------------------
# Rare categories across meals are weighted higher.
cat_counts = Counter()
for cats in meal_to_cats.values():
    cat_counts.update(cats)

M = len(all_meals)
DEFAULT_W = 1.0
cat_weight = {c: (math.log(M / (cat_counts[c] + 1)) + 1) for c in cat_counts}

print("\n=== Category Weight Samples ===")
print("Most common categories:")
for c, cnt in cat_counts.most_common(8):
    print(f"  {c:30s} freq={cnt:4d}  w={cat_weight.get(c, DEFAULT_W):.3f}")

print("\nSome rare categories:")
for c, cnt in sorted(cat_counts.items(), key=lambda x: x[1])[:8]:
    print(f"  {c:30s} freq={cnt:4d}  w={cat_weight.get(c, DEFAULT_W):.3f}")



=== Category Weight Samples ===
Most common categories:
  Salad                          freq= 435  w=2.523
  Soup                           freq= 422  w=2.554
  Dessert                        freq= 418  w=2.563
  Appetizer                      freq= 416  w=2.568
  Beverage                       freq= 412  w=2.577
  Side                           freq= 409  w=2.585
  Main Course                    freq= 406  w=2.592
  Breakfast Item                 freq= 398  w=2.612

Some rare categories:
  Breakfast Item                 freq= 398  w=2.612
  Main Course                    freq= 406  w=2.592
  Side                           freq= 409  w=2.585
  Beverage                       freq= 412  w=2.577
  Appetizer                      freq= 416  w=2.568
  Dessert                        freq= 418  w=2.563
  Soup                           freq= 422  w=2.554
  Salad                          freq= 435  w=2.523


In [8]:
# ----------------------------
# 7) DYNAMIC MARGINAL-UTILITY SOLVER (MEALS)
# ----------------------------
def solve_dynamic_meal_bundle(
    required_categories,         # set[str]
    K=3,
    base=0.5,
    cost_constant=0.0
):
    """
    Pick up to K meals to maximize diminishing-returns coverage of required_categories.
    Diminishing returns: w(c) * base^(count_already_covered)
    """
    team = []
    coverage_counts = {c: 0 for c in required_categories}

    while len(team) < K:
        best_meal, best_mu = None, 0.0

        for meal, cats in meal_to_cats.items():
            if meal in team:
                continue

            cats_hit = cats & required_categories
            util = 0.0
            for c in cats_hit:
                n = coverage_counts[c]
                util += cat_weight.get(c, DEFAULT_W) * (base ** n)

            mu = util - cost_constant
            if mu > best_mu:
                best_mu = mu
                best_meal = meal

        if best_meal is None or best_mu <= 0:
            break

        team.append(best_meal)
        for c in (meal_to_cats[best_meal] & required_categories):
            coverage_counts[c] += 1

    covered = set().union(*[(meal_to_cats[m] & required_categories) for m in team]) if team else set()

    debug = {
        "covered_categories": covered,
        "coverage_pct": coverage_pct(required_categories, covered),
        "coverage_counts": coverage_counts,
    }
    return team, debug


In [9]:
# ----------------------------
# 8) RUN FOR ALL USERS/ROWS + ULTRA METRIC + SAVE
# ----------------------------
print("\n=== Running dynamic bundle selection for each user row ===")
rows = []

for idx, row in req_df.iterrows():
    user_id = row[USER_ID_COL]
    occasion = row[OCCASION_COL]
    req_set = row["req_set"]

    team, dbg = solve_dynamic_meal_bundle(
        required_categories=req_set,
        K=K_MEALS,
        base=DIM_BASE,
        cost_constant=COST_CONSTANT
    )

    # Calculate Ultra Goodness
    goodness_ultra = M1.apply_ultra_metric(req_set, team, meal_to_cats)

    # UPDATED DICTIONARY TO MATCH YOUR REQUESTED FORMAT
    rows.append({
        "user_id": user_id,
        "meal_occasion": occasion,
        "lead_meal": team[0] if team else None,  # First meal is the lead
        "meal_bundle": team,                     # Full list of meals
        "goodness_score": goodness_ultra         # Final metric
    })

    # Print a few examples
    if idx < 3:
        print(f"\n--- Example {idx+1} ---")
        print("User:", user_id, "| Occasion:", occasion)
        print("Team (Bundle):", team)
        print("Ultra Goodness Score:", round(goodness_ultra, 4))

    if (idx + 1) % 50 == 0:
        print(f"Processed {idx+1}/{len(req_df)} rows...")

# Convert to DataFrame
results_df = pd.DataFrame(rows)

# Select and order the specific columns you requested
final_output_df = results_df[["user_id", "meal_occasion", "lead_meal", "meal_bundle", "goodness_score"]]

print("\n=== Results preview ===")
print(final_output_df.head(10).to_string(index=False))

# Save the file
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = f"{OUT_DIR}/marginal_utility_meal_results_{timestamp}.csv"
final_output_df.to_csv(out_csv, index=False)

print("\nSaved output to:", out_csv)


=== Running dynamic bundle selection for each user row ===

--- Example 1 ---
User: user_0_breakfast | Occasion: breakfast
Team (Bundle): ['Milk Spicy (m16)', 'Cereal Special (m78)', 'Chicken Skewers Spicy (m100)']
Ultra Goodness Score: 0.575

--- Example 2 ---
User: user_1_breakfast | Occasion: breakfast
Team (Bundle): ['Cereal Special (m78)', 'Mango Juice Spicy (m233)', 'Breakfast Sandwich Special (m898)']
Ultra Goodness Score: 0.575

--- Example 3 ---
User: user_2_breakfast | Occasion: breakfast
Team (Bundle): ['Milk Spicy (m16)', 'Cereal Special (m78)', 'Chicken Skewers Spicy (m100)']
Ultra Goodness Score: 0.575
Processed 50/500 rows...
Processed 100/500 rows...
Processed 150/500 rows...
Processed 200/500 rows...
Processed 250/500 rows...
Processed 300/500 rows...
Processed 350/500 rows...
Processed 400/500 rows...
Processed 450/500 rows...
Processed 500/500 rows...

=== Results preview ===
         user_id meal_occasion                    lead_meal                                